# 🌡️ RNN Assignment: Weather Temperature Forecasting with a Vanilla RNN
### Learning the core recurrent computation — no gates, no shortcuts

---

## 🎯 Learning Objectives
By the end of this assignment you will be able to:
1. Explain and implement the **vanilla RNN forward equation** from scratch in NumPy and PyTorch
2. Understand **weight sharing across time** and why it works
3. Perform professional **EDA on time-series data** including stationarity checks
4. Engineer **temporal features** (lags, rolling statistics, cyclical encodings)
5. Implement a **proper training loop** with gradient clipping, early stopping, and learning-rate scheduling
6. Analyse the **vanishing gradient problem** numerically — why vanilla RNNs struggle with long sequences
7. **Evaluate** with domain-appropriate metrics and visualise forecasts
8. **Package** the trained model for deployment

---

## 📋 Problem Statement

**Your task:** Build a Vanilla RNN model that predicts **tomorrow's maximum temperature (°C)**
given the previous **14 days** of weather observations.

This is a short-horizon forecasting problem — ideal for a vanilla RNN because the
dependencies are close enough in time that vanishing gradients are manageable.

---

## 📖 How to Use This Notebook

| What you see | What to do |
|---|---|
| Narrative text (this) | Read carefully |
| ✈️ Running example cells | Read + run — these are **fully implemented** |
| `# ── TODO` cells | **Write your code here** |
| 💡 boxes | Industry tips — important context |
| ⚠️ boxes | Common mistakes to avoid |

> The running example uses **sine wave prediction** — the classic toy problem for teaching RNNs.
> Every pattern in the example maps directly to the assignment.

---

## 🗂️ Notebook Structure

| Section | Topic | Role |
|---------|-------|------|
| 0 | Setup | Both |
| ✈️ Running example | Sine wave forecasting, full pipeline | Read & run |
| 1 | Data acquisition | **You implement** |
| 2 | EDA | **You implement** |
| 3 | Feature engineering | **You implement** |
| 4 | Preprocessing + sequences | **You implement** |
| 5 | Vanilla RNN architecture | **You implement** |
| 6 | Training loop | **You implement** |
| 7 | Gradient analysis | **You implement** |
| 8 | Evaluation | **You implement** |
| 9 | Hyperparameter tuning | **You implement** |
| 10 | Deployment | **You implement** |
| 11 | Reflection | **You write** |

---
## ⚙️ Step 0 — Environment Setup

Run this cell first. It installs packages and sets random seeds.

In [ ]:
# ── Install & import ──────────────────────────────────────────────────────────
!pip install openmeteo-requests requests-cache retry-requests --quiet

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.dates as mdates
import warnings
warnings.filterwarnings("ignore")

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

import openmeteo_requests, requests_cache
from retry_requests import retry
from datetime import datetime, timedelta
import math, json, time, pickle, os, random

# ── Reproducibility ────────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device  : {device}")
print(f"PyTorch : {torch.__version__}")
print("Setup complete ✓")

---
---
# ✈️ Running Example: Sine Wave Sequence Prediction

> **Read and run every cell in this section before starting the assignment.**
> All the patterns you need are demonstrated here.

---

## Why sine waves?

The sine wave is the canonical toy problem for teaching RNNs because:
- The sequence is **perfectly periodic** — the RNN must remember the phase
- There is **one true latent state** (phase) that must be tracked through time
- You can visualise predictions against the ground truth trivially
- Training is fast — you can see results in under 30 seconds

**Task:** Given the last `SEQ_LEN = 20` values of a noisy sine wave,
predict the **next value** one step ahead.

---

### The Vanilla RNN Equation (the ONLY equation we use)

At every time step `t`, the RNN applies:

```
h_t  =  tanh( W_xh · x_t  +  W_hh · h_{t-1}  +  b_h )   ← update hidden state
y_t  =  W_hy · h_t  +  b_y                                ← read prediction
```

**That is the entire model.** No forget gates. No cell states. No attention.
Just two matrix multiplications, an addition, and a tanh — repeated for every time step.

In [ ]:
# ══ EXAMPLE ▸ Step 1: Generate Sine Wave Data ═══════════════════════════════
np.random.seed(42)

# Parameters
N_POINTS = 2000       # total data points
AMPLITUDE = 1.0
FREQ      = 0.05      # cycles per sample
NOISE_STD = 0.08      # Gaussian noise added

t = np.arange(N_POINTS)
sine_clean = AMPLITUDE * np.sin(2 * np.pi * FREQ * t)
sine_noisy = sine_clean + np.random.normal(0, NOISE_STD, N_POINTS)

print(f"Dataset size : {N_POINTS} samples")
print(f"Frequency    : {FREQ} cycles/sample  →  period = {int(1/FREQ)} samples")
print(f"Amplitude    : {AMPLITUDE}")
print(f"Noise std    : {NOISE_STD}")
print(f"Signal range : [{sine_noisy.min():.3f}, {sine_noisy.max():.3f}]")

# ── Quick look ─────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 3.5))

ax = axes[0]
ax.plot(t[:200], sine_noisy[:200], color="steelblue", lw=1.2, label="Noisy signal")
ax.plot(t[:200], sine_clean[:200], color="red",       lw=1.8, ls="--", alpha=0.7, label="Clean sine")
ax.set_title("Noisy Sine Wave (first 200 samples)", fontsize=12, fontweight="bold")
ax.set_xlabel("Time step t"); ax.set_ylabel("Amplitude")
ax.legend(); ax.grid(alpha=0.3)

ax = axes[1]
# Lag-1 autocorrelation plot — confirms strong sequential structure
lags = range(1, 41)
autocorr = [pd.Series(sine_noisy).autocorr(lag) for lag in lags]
ax.bar(lags, autocorr, color="coral", alpha=0.8)
ax.axhline(0, color="black", lw=0.8)
ax.set_title("Autocorrelation of Noisy Sine\n(confirms sequential structure)", fontsize=11)
ax.set_xlabel("Lag"); ax.set_ylabel("Autocorrelation")
ax.grid(alpha=0.3, axis="y")

plt.tight_layout(); plt.show()
print("Autocorrelation is high for small lags → past values predict future values ✓")

In [ ]:
# ══ EXAMPLE ▸ Step 2: Preprocessing & Sequence Creation ══════════════════════

SEQ_LEN_EX = 20    # use last 20 values to predict next 1

# ── Scale to [-1, 1] ──────────────────────────────────────────────────────────
# IMPORTANT: fit scaler on TRAINING DATA ONLY to prevent data leakage
train_frac = 0.80
n_train_ex = int(N_POINTS * train_frac)

scaler_ex = MinMaxScaler(feature_range=(-1, 1))
train_data_ex = sine_noisy[:n_train_ex].reshape(-1, 1)
test_data_ex  = sine_noisy[n_train_ex:].reshape(-1, 1)

scaler_ex.fit(train_data_ex)                         # ← fit on train ONLY
train_scaled_ex = scaler_ex.transform(train_data_ex).flatten()
test_scaled_ex  = scaler_ex.transform(test_data_ex).flatten()

print(f"Train samples : {len(train_scaled_ex)}  ({train_frac:.0%})")
print(f"Test  samples : {len(test_scaled_ex)}   ({1-train_frac:.0%})")
print(f"Scaled range  : [{train_scaled_ex.min():.3f}, {train_scaled_ex.max():.3f}]")

# ── Sliding-window sequence creation ─────────────────────────────────────────
def make_sequences_1d(data, seq_len):
    """
    Convert a 1D array into overlapping (X, y) pairs.

    Each X is a window of length seq_len (shape: seq_len × 1).
    Each y is the single next value after that window.

    Returns:
        X : (n_samples, seq_len, 1)   ← PyTorch expects (batch, seq, features)
        y : (n_samples,)
    """
    X, y = [], []
    for i in range(len(data) - seq_len):
        X.append(data[i : i + seq_len])        # look-back window
        y.append(data[i + seq_len])             # next value to predict
    return (np.array(X, dtype=np.float32).reshape(-1, seq_len, 1),
            np.array(y, dtype=np.float32))

X_train_ex, y_train_ex = make_sequences_1d(train_scaled_ex, SEQ_LEN_EX)
X_test_ex,  y_test_ex  = make_sequences_1d(test_scaled_ex,  SEQ_LEN_EX)

print(f"\nX_train : {X_train_ex.shape}  →  (samples, seq_len, n_features)")
print(f"y_train : {y_train_ex.shape}  →  (samples,)")
print(f"X_test  : {X_test_ex.shape}")
print(f"\nEach input  = {SEQ_LEN_EX} consecutive sine values")
print(f"Each target = the value at position t+1")

In [ ]:
# ══ EXAMPLE ▸ Step 3: PyTorch Dataset & DataLoader ══════════════════════════

class SequenceDataset(Dataset):
    """Wraps (X, y) numpy arrays for PyTorch DataLoader."""
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)
    def __len__(self):
        return len(self.X)
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

BATCH_SIZE_EX = 32

train_ds_ex = SequenceDataset(X_train_ex, y_train_ex)
test_ds_ex  = SequenceDataset(X_test_ex,  y_test_ex)
train_dl_ex = DataLoader(train_ds_ex, batch_size=BATCH_SIZE_EX, shuffle=True,  drop_last=True)
test_dl_ex  = DataLoader(test_ds_ex,  batch_size=BATCH_SIZE_EX, shuffle=False)

# Sanity check
xb, yb = next(iter(train_dl_ex))
print(f"Batch X : {xb.shape}  →  (batch_size, seq_len, n_features)")
print(f"Batch y : {yb.shape}  →  (batch_size,)")
print(f"X range : [{xb.min():.3f}, {xb.max():.3f}]")
print(f"y range : [{yb.min():.3f}, {yb.max():.3f}]")

In [ ]:
# ══ EXAMPLE ▸ Step 4: Vanilla RNN Architecture ═══════════════════════════════
#
# THE CORE EQUATION (implemented inside nn.RNN):
#   h_t = tanh( W_xh · x_t  +  W_hh · h_{t-1}  +  b_h )
#   y_t = W_hy · h_t  +  b_y
#
# No forget gates. No input gates. No cell state.
# Just tanh + two linear transformations, repeated T times.

class VanillaRNN(nn.Module):
    """
    Vanilla (Elman) RNN for one-step-ahead sequence forecasting.

    Architecture:
        input (n_features)
        → nn.RNN  [applies h_t = tanh(W_xh·x_t + W_hh·h_{t-1} + b_h)]
        → Dropout
        → nn.Linear  [y_t = W_hy·h_t + b_y]
        → scalar output

    Args:
        input_size  : number of features per time step (1 for univariate)
        hidden_size : dimension of the hidden state vector h_t
        num_layers  : stacked RNN layers (depth)
        dropout     : dropout probability (applied between RNN layers if num_layers>1)
        output_size : prediction horizon (1 = next step only)
    """
    def __init__(self, input_size=1, hidden_size=32, num_layers=1,
                 dropout=0.0, output_size=1):
        super(VanillaRNN, self).__init__()
        self.hidden_size = hidden_size
        self.num_layers  = num_layers

        # nn.RNN implements the vanilla Elman recurrent equation
        # nonlinearity='tanh' is default and must NOT be changed for vanilla RNN
        self.rnn = nn.RNN(
            input_size   = input_size,
            hidden_size  = hidden_size,
            num_layers   = num_layers,
            nonlinearity = "tanh",      # ← defines vanilla RNN (not LSTM, not GRU)
            batch_first  = True,        # input shape: (batch, seq, feature)
            dropout      = dropout if num_layers > 1 else 0.0
        )

        # Regularisation between RNN and output
        self.dropout = nn.Dropout(dropout)

        # Output layer: maps final hidden state → scalar prediction
        self.fc = nn.Linear(hidden_size, output_size)

    def forward(self, x, h0=None):
        """
        Args:
            x  : (batch, seq_len, input_size)
            h0 : (num_layers, batch, hidden_size)  optional initial hidden state
        Returns:
            pred   : (batch,)     — predicted next value
            hidden : (num_layers, batch, hidden_size)  — final hidden state
        """
        # RNN processes the full sequence, returns all hidden states + final state
        out, hidden = self.rnn(x, h0)
        # out shape: (batch, seq_len, hidden_size)
        # We only use the LAST time step's hidden state for prediction
        last_hidden = out[:, -1, :]             # (batch, hidden_size)
        last_hidden = self.dropout(last_hidden)
        pred = self.fc(last_hidden).squeeze(-1) # (batch,)
        return pred, hidden

    def init_hidden(self, batch_size, device):
        """Zero-initialise hidden state. Called at the start of each sequence batch."""
        return torch.zeros(self.num_layers, batch_size, self.hidden_size,
                           device=device)


# Instantiate
model_ex = VanillaRNN(input_size=1, hidden_size=32, num_layers=1,
                      dropout=0.0, output_size=1).to(device)

print(model_ex)
n_params = sum(p.numel() for p in model_ex.parameters())
print(f"\nTotal parameters : {n_params:,}")
print("\nParameter breakdown:")
for name, param in model_ex.named_parameters():
    print(f"  {name:30s}  shape={str(param.shape):<20}  n={param.numel()}")

# Quick forward pass test
with torch.no_grad():
    out_test, h_test = model_ex(xb.to(device))
    print(f"\nForward pass → output shape: {out_test.shape}  (should be [{BATCH_SIZE_EX}])")
    print(f"Hidden state shape          : {h_test.shape}")

In [ ]:
# ══ EXAMPLE ▸ Step 5: Training Loop ══════════════════════════════════════════
#
# Industry-standard training loop with:
#   ✓ MSE loss (appropriate for regression)
#   ✓ Adam optimiser
#   ✓ ReduceLROnPlateau scheduler (halves LR when val loss plateaus)
#   ✓ Gradient clipping — CRITICAL for vanilla RNN (prevents exploding gradients)
#   ✓ Early stopping (patience-based — avoids overfitting)
#   ✓ Best model checkpointing

def train_rnn(model, train_loader, val_loader, n_epochs=100,
              lr=1e-3, patience=15, clip_norm=1.0, device="cpu"):
    """
    Trains a VanillaRNN (or any nn.Module with the same interface).

    Key parameters:
        clip_norm : max gradient norm — essential for vanilla RNNs
                    (LSTM has built-in gating that reduces explosion risk;
                     vanilla RNN does NOT — clipping is mandatory)
        patience  : stop if val loss doesn't improve for this many epochs
    """
    criterion = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="min", factor=0.5, patience=7, verbose=False)

    train_losses, val_losses = [], []
    best_val = float("inf")
    best_state = None
    wait = 0

    for epoch in range(1, n_epochs + 1):
        # ── Training ────────────────────────────────────────────────────────
        model.train()
        epoch_loss = 0.0
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            pred, _ = model(xb)
            loss     = criterion(pred, yb)
            loss.backward()

            # !! Gradient clipping — do NOT remove for vanilla RNN !!
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=clip_norm)

            optimizer.step()
            epoch_loss += loss.item() * len(xb)

        avg_train = epoch_loss / len(train_loader.dataset)

        # ── Validation ──────────────────────────────────────────────────────
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for xb, yb in val_loader:
                xb, yb = xb.to(device), yb.to(device)
                pred, _ = model(xb)
                val_loss += criterion(pred, yb).item() * len(xb)
        avg_val = val_loss / len(val_loader.dataset)

        train_losses.append(avg_train)
        val_losses.append(avg_val)
        scheduler.step(avg_val)

        # ── Early stopping & checkpointing ───────────────────────────────────
        if avg_val < best_val:
            best_val   = avg_val
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            wait = 0
        else:
            wait += 1

        if wait >= patience:
            print(f"  Early stop at epoch {epoch}  (patience={patience})")
            break

        if epoch % 10 == 0 or epoch == 1:
            lr_now = optimizer.param_groups[0]["lr"]
            print(f"  Epoch {epoch:3d}  train={avg_train:.6f}  "
                  f"val={avg_val:.6f}  lr={lr_now:.2e}")

    model.load_state_dict(best_state)
    print(f"\n  Best val MSE: {best_val:.6f}  (RMSE={math.sqrt(best_val):.5f})")
    return train_losses, val_losses


print("Training VanillaRNN on sine wave...")
print("-" * 60)
t0 = time.time()
train_l_ex, val_l_ex = train_rnn(
    model_ex, train_dl_ex, test_dl_ex,
    n_epochs=120, lr=1e-3, patience=20,
    clip_norm=1.0, device=str(device)
)
print(f"Elapsed: {time.time()-t0:.1f}s")

# ── Plot training curves ──────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(train_l_ex, label="Train MSE", color="steelblue", lw=2)
ax.plot(val_l_ex,   label="Val MSE",   color="coral",     lw=2)
ax.set_xlabel("Epoch"); ax.set_ylabel("MSE Loss")
ax.set_title("Training Curve — Sine Wave VanillaRNN", fontsize=12, fontweight="bold")
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

In [ ]:
# ══ EXAMPLE ▸ Step 6: Evaluation ═════════════════════════════════════════════

def get_predictions(model, loader, scaler, device):
    """Run inference on a DataLoader, inverse-transform to original scale."""
    model.eval()
    preds_s, trues_s = [], []
    with torch.no_grad():
        for xb, yb in loader:
            pred, _ = model(xb.to(device))
            preds_s.append(pred.cpu().numpy())
            trues_s.append(yb.numpy())

    preds_s = np.concatenate(preds_s).reshape(-1, 1)
    trues_s = np.concatenate(trues_s).reshape(-1, 1)

    # Inverse-transform back to original amplitude
    preds_orig = scaler.inverse_transform(preds_s).flatten()
    trues_orig = scaler.inverse_transform(trues_s).flatten()
    return preds_orig, trues_orig

preds_ex, trues_ex = get_predictions(model_ex, test_dl_ex, scaler_ex, device)

rmse_ex = math.sqrt(mean_squared_error(trues_ex, preds_ex))
mae_ex  = mean_absolute_error(trues_ex, preds_ex)
r2_ex   = r2_score(trues_ex, preds_ex)

print("TEST METRICS (original scale)")
print(f"  RMSE : {rmse_ex:.5f}")
print(f"  MAE  : {mae_ex:.5f}")
print(f"  R²   : {r2_ex:.5f}")
print(f"\n  Noise std was {NOISE_STD:.2f} — a good model should have RMSE ≈ noise level")

# ── Forecast plot ─────────────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 1, figsize=(14, 8))

ax = axes[0]
n_show = 200
ax.plot(trues_ex[:n_show], color="steelblue", lw=2,   label="Actual")
ax.plot(preds_ex[:n_show], color="coral",     lw=1.8, ls="--", label="Predicted")
ax.fill_between(range(n_show),
                preds_ex[:n_show] - rmse_ex,
                preds_ex[:n_show] + rmse_ex,
                alpha=0.12, color="coral", label="±RMSE band")
ax.set_title(f"Sine Wave Forecast — first {n_show} test samples  "
             f"(RMSE={rmse_ex:.4f}, R²={r2_ex:.4f})",
             fontsize=12, fontweight="bold")
ax.set_ylabel("Amplitude"); ax.legend(); ax.grid(alpha=0.3)

ax = axes[1]
residuals = trues_ex - preds_ex
ax.scatter(range(len(residuals)), residuals, s=6, alpha=0.5,
           c=["coral" if r < 0 else "steelblue" for r in residuals])
ax.axhline(0, color="black", lw=1)
ax.set_title("Residuals — should be random (white noise)", fontsize=11)
ax.set_xlabel("Test sample index"); ax.set_ylabel("Residual")
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig("example_eval.png", dpi=110, bbox_inches="tight")
plt.show()

In [ ]:
# ══ EXAMPLE ▸ Step 7: The Vanishing Gradient — Key Vanilla RNN Limitation ═══
#
# This cell demonstrates numerically WHY vanilla RNNs struggle with long sequences.
# Understanding this is the central theoretical insight of this assignment.
#
# The gradient of the loss w.r.t. h_0 (initial hidden state) involves:
#   ∂h_t/∂h_0  =  ∏_{k=1}^{t}  diag(tanh'(z_k)) · W_hh
#
# The spectral radius of W_hh and the magnitude of tanh'(z_k) ≤ 1
# together determine whether gradients vanish or explode.

print("=" * 60)
print("VANISHING GRADIENT ANALYSIS")
print("=" * 60)
print()
print("tanh'(z) = 1 - tanh²(z)  ∈ (0, 1]")
print()
print(f"  {'h value':<12} {'tanh(z)':<12} {'tanh\'(z)':<14} {'after 10 steps':<16} {'after 30 steps'}")
print("  " + "-"*65)
for h_val in [0.0, 0.3, 0.6, 0.9, 0.99]:
    tanh_val = math.tanh(h_val)
    dtanh    = 1 - tanh_val**2
    print(f"  {h_val:<12.2f} {tanh_val:<12.4f} {dtanh:<14.6f} "
          f"{dtanh**10:<16.8f} {dtanh**30:.12f}")

print()
print("Observation: if hidden units saturate (|h| → 1), tanh'(z) → 0,")
print("and gradients vanish in O(T) steps — the model forgets early inputs.")

# ── Visualise gradient magnitude across time steps ────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: gradient decay for different saturation levels
ax = axes[0]
seq_lengths = np.arange(1, 51)
for h_val, col, label in [(0.0, "green",  "h=0.00 (not saturated)"),
                           (0.5, "blue",   "h=0.50"),
                           (0.8, "orange", "h=0.80"),
                           (0.95,"red",    "h=0.95 (near-saturated)")]:
    dtanh = 1 - math.tanh(h_val)**2
    grad_mag = [dtanh**t for t in seq_lengths]
    ax.semilogy(seq_lengths, grad_mag, "-o", ms=3, lw=2, color=col, label=label)

ax.axhline(1e-4, color="gray", ls="--", lw=1, alpha=0.7, label="≈ 0 (1e-4 threshold)")
ax.set_xlabel("Time steps backward")
ax.set_ylabel("Gradient magnitude (log scale)")
ax.set_title("Vanishing Gradient in Vanilla RNN\n∂h_t/∂h_0 ≈ tanh\'(z)^T",
             fontsize=11, fontweight="bold")
ax.legend(fontsize=8); ax.grid(alpha=0.3)

# Right: compare vanilla RNN vs LSTM gradient flow (illustrative)
ax = axes[1]
steps = np.arange(1, 51)
# Vanilla RNN: rapid decay (approximate)
rnn_grad  = 0.85**steps
# LSTM: slow decay because forget gate can be ~1 (additive cell state)
lstm_grad = np.clip(0.99**steps + 0.01, 0, 1)

ax.semilogy(steps, rnn_grad,  "r-o", ms=3, lw=2, label="Vanilla RNN (approx.)")
ax.semilogy(steps, lstm_grad, "g-o", ms=3, lw=2, label="LSTM (approx.)")
ax.axhline(1e-4, color="gray", ls="--", lw=1, alpha=0.7)
ax.set_xlabel("Time steps backward")
ax.set_ylabel("Gradient magnitude (log scale)")
ax.set_title("Vanilla RNN vs LSTM: Gradient Flow\n(LSTM solves vanishing via additive Cₜ update)",
             fontsize=11, fontweight="bold")
ax.legend(fontsize=9); ax.grid(alpha=0.3)

plt.suptitle("The Vanishing Gradient Problem — Why This Matters for Model Design",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("vanishing_gradient.png", dpi=110, bbox_inches="tight")
plt.show()

print()
print("KEY TAKEAWAY for this assignment:")
print("  Vanilla RNN works well when dependencies are SHORT (≤ 15-20 steps).")
print("  Weather prediction with a 14-day lookback is a safe regime.")
print("  Longer lookbacks would require LSTM/GRU to avoid gradient death.")

In [ ]:
# ══ EXAMPLE ▸ Step 8: Deployment Packaging ════════════════════════════════════
# Industry practice: bundle model + scaler + config into a deployable object.

class SineForecasterDeployment:
    """
    Self-contained predictor ready for deployment.
    Wraps: trained VanillaRNN + scaler + config.
    Exposes a clean .predict() interface for inference.
    """
    def __init__(self, model, scaler, config: dict):
        self.model  = model
        self.scaler = scaler
        self.config = config

    @torch.no_grad()
    def predict(self, recent_values: np.ndarray) -> float:
        """
        Predict next value from a window of recent observations.

        Args:
            recent_values : 1D numpy array of length seq_len (original scale)
        Returns:
            float — predicted next value (original scale)
        """
        seq_len = self.config["seq_len"]
        assert len(recent_values) == seq_len, \
               f"Expected {seq_len} values, got {len(recent_values)}"

        # Scale → sequence tensor → forward pass → inverse scale
        scaled = self.scaler.transform(recent_values.reshape(-1, 1)).flatten()
        x = torch.tensor(scaled, dtype=torch.float32).unsqueeze(0).unsqueeze(-1)  # (1,T,1)
        self.model.eval()
        pred_s, _ = self.model(x)
        return float(self.scaler.inverse_transform([[pred_s.item()]])[0, 0])

    def save(self, prefix="sine_model"):
        torch.save(self.model.state_dict(), f"{prefix}_weights.pt")
        with open(f"{prefix}_scaler.pkl", "wb") as f:
            pickle.dump(self.scaler, f)
        with open(f"{prefix}_config.json", "w") as f:
            json.dump(self.config, f, indent=2)
        print(f"Saved: {prefix}_weights.pt | _scaler.pkl | _config.json")

    @classmethod
    def load(cls, prefix, device="cpu"):
        with open(f"{prefix}_config.json") as f:
            cfg = json.load(f)
        m = VanillaRNN(input_size=1, hidden_size=cfg["hidden_size"],
                       num_layers=cfg["num_layers"]).to(device)
        m.load_state_dict(torch.load(f"{prefix}_weights.pt", map_location=device))
        with open(f"{prefix}_scaler.pkl", "rb") as f:
            sc = pickle.load(f)
        return cls(m, sc, cfg)

# ── Create, save, and test ─────────────────────────────────────────────────────
config_sine = {
    "seq_len"     : SEQ_LEN_EX,
    "hidden_size" : model_ex.hidden_size,
    "num_layers"  : model_ex.num_layers,
    "trained_on"  : "Sine wave (A=1, freq=0.05, noise=0.08)",
    "created_at"  : datetime.now().isoformat(),
    "test_rmse"   : round(rmse_ex, 6),
}

deployer_ex = SineForecasterDeployment(model_ex, scaler_ex, config_sine)
deployer_ex.save("sine_model")

# Simulate inference (last window of sine wave → predict next)
last_window = sine_noisy[-SEQ_LEN_EX:]
predicted   = deployer_ex.predict(last_window)
print(f"\nInference call:")
print(f"  Input   : last {SEQ_LEN_EX} sine values  ({last_window[-3:].round(3)} ...)")
print(f"  Output  : {predicted:.5f}  (predicted next value)")
print(f"  Expected: ~{sine_clean[-1]:.5f}  (true next clean value)")

# Load from disk and verify round-trip
deployer_ex2 = SineForecasterDeployment.load("sine_model", str(device))
pred2 = deployer_ex2.predict(last_window)
print(f"\nLoad-from-disk check:")
print(f"  Original : {predicted:.8f}")
print(f"  Reloaded : {pred2:.8f}")
print(f"  Delta    : {abs(predicted - pred2):.2e}  (should be ~0)")
print("\n✅ Running example complete. Now start the assignment below.")

---
---
# 📝 Assignment: Daily Max Temperature Forecasting with a Vanilla RNN

> **Your turn.** Following the same steps as the running example,
> build a Vanilla RNN model to predict **tomorrow's maximum temperature**
> for a city of your choice.

---

## 📋 Dataset
You will use **Open-Meteo** (free, no API key) to download historical daily
maximum temperature for any major city. We will use **New York City** as default.

Source: https://open-meteo.com/  
Data: Daily `temperature_2m_max` (°C) from 2015-01-01 to 2023-12-31

## 🎯 Target
Predict **tomorrow's daily maximum temperature (°C)** given the last **14 days**.

## ✅ Grading Rubric
| Criterion | Marks |
|-----------|-------|
| Data acquisition & correct inspection | 10 |
| EDA: ≥ 4 plots with interpretations | 20 |
| Feature engineering (≥ 4 features) | 15 |
| Correct preprocessing (no leakage) | 10 |
| Vanilla RNN architecture (correct equation) | 15 |
| Training loop with gradient clipping | 10 |
| Gradient analysis section | 5 |
| Evaluation metrics + plots | 10 |
| Deployment packaging | 5 |
| Reflection questions | 5 (bonus) |

---
## 📝 Step 1 — Data Acquisition & Initial Inspection

Download daily weather data for New York City (or another city of your choice)
from Open-Meteo and perform an initial inspection.

**Required output:**
- Shape, date range, number of observations
- Column names and dtypes
- Missing value count
- Basic descriptive statistics for `temperature_2m_max`

> 💡 **Industry note:** Always check for missing values and data quality issues
> before any modelling. Missing weather data (station outages) is common and
> must be handled explicitly — not silently dropped.

> ⚠️ **Common mistake:** Do not compute descriptive statistics on the
> un-inspected data. Always look at the head/tail first.

In [ ]:
# ── TODO ─────────────────────────────────────────────────────────────
# ── Step 1: Download and inspect weather data ─────────────────────────────────
import openmeteo_requests, requests_cache
from retry_requests import retry

# TODO 1a: Set up the Open-Meteo API client (copy from docs pattern)
# Use requests_cache with expire_after=3600 and retry on error
# YOUR CODE HERE

# TODO 1b: Download data for New York City
# Parameters:
#   latitude=40.7128, longitude=-74.0060
#   start_date="2015-01-01", end_date="2023-12-31"
#   daily=["temperature_2m_max", "temperature_2m_min", "precipitation_sum",
#           "windspeed_10m_max", "shortwave_radiation_sum"]
# YOUR CODE HERE

# TODO 1c: Convert to a pandas DataFrame with a datetime index
# Column names: temperature_2m_max, temperature_2m_min, precipitation_sum,
#               windspeed_10m_max, shortwave_radiation_sum
# YOUR CODE HERE

# TODO 1d: Print inspection results
# - df.shape
# - df.dtypes
# - df.head() and df.tail()
# - df.isnull().sum()
# - df["temperature_2m_max"].describe()
# YOUR CODE HERE
raise NotImplementedError("Fill in the TODO above before continuing.")

---
## 📝 Step 2 — Exploratory Data Analysis (EDA)

Produce **at least 4** well-labelled plots. After each plot write a one-sentence interpretation
as a code comment.

**Suggested plots (pick at least 4):**

| Plot | What to look for |
|------|-----------------|
| Full temperature time series | Seasonal pattern, trend |
| Seasonal decomposition (monthly averages) | Amplitude of seasons |
| Rolling mean and std | Trend + volatility regime |
| Temperature vs solar radiation scatter | Predictor correlation |
| Autocorrelation function (ACF) | How many lags matter |
| Distribution of daily max temperature | Shape, tails |
| Year-on-year overlay | Consistency of seasonality |

> 💡 **Why ACF matters:** The autocorrelation function tells you how many lag
> values are informative. If ACF drops to near-zero after 14 lags, then
> `SEQ_LEN = 14` is justified. If it stays high for 30+ lags, a longer window may help.

In [ ]:
# ── TODO ─────────────────────────────────────────────────────────────
# ── Step 2: EDA — at least 4 plots ───────────────────────────────────────────

# TODO 2a: Time series plot of temperature_2m_max (full date range)
# YOUR CODE HERE

# TODO 2b: Monthly average temperature (bar chart — reveals seasonality)
# YOUR CODE HERE

# TODO 2c: Rolling 30-day mean and std of temperature
# YOUR CODE HERE

# TODO 2d: Autocorrelation plot for temperature_2m_max (lags 1–40)
# YOUR CODE HERE

# TODO 2e: (Optional) Any additional plot you find informative
# YOUR CODE HERE

# After all plots, write:
# EDA Finding 1:
# EDA Finding 2:
# EDA Finding 3:
raise NotImplementedError("Fill in the TODO above before continuing.")

---
## 📝 Step 3 — Feature Engineering

Create temporal features that will help the vanilla RNN model. Include at least:

| Feature | Formula | Why useful |
|---------|---------|------------|
| `temp_lag1` | `temperature_2m_max.shift(1)` | Yesterday's temp (strongest predictor) |
| `temp_lag7` | `temperature_2m_max.shift(7)` | Same day last week |
| `roll_mean7` | 7-day rolling mean of max temp | Short-term trend |
| `roll_std7` | 7-day rolling std of max temp | Recent variability |
| `day_sin` | `sin(2π · day_of_year / 365)` | Cyclical annual encoding |
| `day_cos` | `cos(2π · day_of_year / 365)` | Cyclical annual encoding |
| `temp_range` | `temperature_2m_max − temperature_2m_min` | Daily thermal range |

> ⚠️ **Why sin/cos for day of year?**
> Day 365 and Day 1 are adjacent in reality but 364 apart numerically.
> Sine/cosine encoding makes them close in feature space — essential for
> seasonal models. A model given `day_of_year` as a raw integer will treat
> Dec 31 and Jan 1 as maximally different, which is wrong.

> ⚠️ **Lagged features and NaN:** `shift()` and `rolling()` create NaN in the
> first rows. Always drop NaN rows BEFORE splitting.

In [ ]:
# ── TODO ─────────────────────────────────────────────────────────────
# ── Step 3: Feature engineering ───────────────────────────────────────────────

df_feat = df.copy()

# TODO 3a: Lag features — yesterday and same day last week
# df_feat["temp_lag1"] = ...
# df_feat["temp_lag7"] = ...
# YOUR CODE HERE

# TODO 3b: Rolling statistics (7-day mean and std of temperature_2m_max)
# YOUR CODE HERE

# TODO 3c: Cyclical encoding of day-of-year
# day_of_year ranges 1..365 (366 in leap years)
# sin encoding: sin(2 * pi * day_of_year / 365)
# cos encoding: cos(2 * pi * day_of_year / 365)
# YOUR CODE HERE

# TODO 3d: Daily temperature range (max - min)
# YOUR CODE HERE

# TODO 3e: Define FEATURE_COLS list and TARGET
# FEATURE_COLS should include temperature_2m_max and all engineered features
# YOUR CODE HERE

# TODO 3f: Drop NaN rows, print shape and feature statistics
# YOUR CODE HERE
raise NotImplementedError("Fill in the TODO above before continuing.")

---
## 📝 Step 4 — Preprocessing & Sequence Creation

Split data, scale features, and create sliding-window sequences.

**Split (chronological — no shuffle):**
- Train: 70% (2015–2021)
- Validation: 15% (2021–2022)
- Test: 15% (2022–2023)

**Sequence length:** `SEQ_LEN = 14` (14 days → predict day 15)

> ⚠️ **No data leakage rules:**
> 1. Split first — then scale
> 2. `scaler.fit()` on training data ONLY
> 3. `scaler.transform()` on all three splits
> 4. Never shuffle time-series data

> 💡 **Multivariate sequences:** Each time step now carries `n_features` values,
> so the input shape is `(batch, seq_len, n_features)` — not `(batch, seq_len, 1)` as in the example.

In [ ]:
# ── TODO ─────────────────────────────────────────────────────────────
# ── Step 4: Preprocessing & Sequence Creation ─────────────────────────────────

SEQ_LEN = 14   # 14-day lookback window

# TODO 4a: Chronological 70/15/15 split (use indices, not shuffle)
# n = len(df_feat)
# YOUR CODE HERE

# TODO 4b: Scale features with MinMaxScaler fitted on train only
# scaler = MinMaxScaler(feature_range=(-1, 1))
# YOUR CODE HERE

# TODO 4c: Create sequences using make_sequences_mv()
# The function below converts a 2D (timesteps, features) array into
# overlapping windows. Implement it:

def make_sequences_mv(data, seq_len, target_idx):
    """
    Create overlapping (X, y) windows from a 2D array.

    Args:
        data       : np.ndarray of shape (T, n_features)
        seq_len    : length of input window
        target_idx : column index in data that is the prediction target

    Returns:
        X : (n_samples, seq_len, n_features) float32
        y : (n_samples,)                     float32  — scaled target values
    """
    # YOUR CODE HERE — build X and y lists, convert to numpy arrays
    pass

# TODO 4d: Call make_sequences_mv for all three splits
# YOUR CODE HERE

# TODO 4e: Create SequenceDataset + DataLoader for all three splits
# batch_size=32, shuffle=True for train, shuffle=False for val and test
# YOUR CODE HERE

# TODO 4f: Print shapes and verify value ranges
# YOUR CODE HERE
raise NotImplementedError("Fill in the TODO above before continuing.")

---
## 📝 Step 5 — Vanilla RNN Architecture

Instantiate the `VanillaRNN` from the running example with multivariate input.

**Requirements:**
- `input_size = n_features` (number of features per time step)
- `hidden_size = 64` (start here; tune in Step 9)
- `num_layers = 1` (single-layer vanilla RNN — keep it simple first)
- `dropout = 0.0` (no dropout for single layer)
- `output_size = 1`

**Print:**
1. Model architecture
2. Named parameter list with shapes and sizes
3. Forward pass output shape

> 💡 **Why `nonlinearity="tanh"`?**
> `nn.RNN` supports two nonlinearities: `"tanh"` and `"relu"`.
> `tanh` is the standard Elman RNN and is what we mean by "vanilla RNN".
> `relu` variants exist but suffer more severely from exploding gradients.
> Always use `tanh` unless you have a specific reason.

In [ ]:
# ── TODO ─────────────────────────────────────────────────────────────
# ── Step 5: Instantiate Vanilla RNN ──────────────────────────────────────────

# TODO 5a: Instantiate VanillaRNN with correct input_size
# YOUR CODE HERE

# TODO 5b: Print model architecture
# YOUR CODE HERE

# TODO 5c: Print named parameters (name, shape, n_params)
# YOUR CODE HERE

# TODO 5d: Verify forward pass — create a dummy tensor and pass it through
# Expected output shape: (batch_size,)
# YOUR CODE HERE
raise NotImplementedError("Fill in the TODO above before continuing.")

---
## 📝 Step 6 — Training Loop

Train your VanillaRNN using the `train_rnn()` function from the running example.

**Requirements:**
1. `n_epochs = 100`, `lr = 1e-3`, `patience = 15`, `clip_norm = 1.0`
2. Plot training and validation loss curves
3. Mark the epoch where early stopping triggered (vertical dashed line)
4. Print best validation loss and corresponding RMSE

> ⚠️ **Gradient clipping is mandatory for vanilla RNNs.**
> Unlike LSTM (which has internal gating that dampens gradient explosion),
> vanilla RNN has no such mechanism. Without clipping, gradients through
> the `W_hh` matrix can grow exponentially over time steps, causing NaN loss.
> The `clip_norm=1.0` argument in `train_rnn()` handles this.

> 💡 **What ReduceLROnPlateau does:**
> If validation loss doesn't improve for `patience=7` epochs, the learning
> rate is halved. This allows the model to continue refining even when the
> initial learning rate is too large to find a good minimum.

In [ ]:
# ── TODO ─────────────────────────────────────────────────────────────
# ── Step 6: Train the model ───────────────────────────────────────────────────

# TODO 6a: Train using train_rnn()
# YOUR CODE HERE

# TODO 6b: Plot training and validation MSE loss curves
# Include: both curves, early stopping vertical line, legend, grid, title
# YOUR CODE HERE

# TODO 6c: Print best validation RMSE
# YOUR CODE HERE
raise NotImplementedError("Fill in the TODO above before continuing.")

---
## 📝 Step 7 — Gradient Magnitude Analysis

Vanilla RNNs are the textbook case for the vanishing gradient problem.
In this step you will **measure it empirically** on your trained model.

**What to do:**
1. After a forward pass, compute gradients w.r.t. the hidden state at each time step
2. Plot how gradient magnitude changes with the step index (from last → first)
3. Compare with the theoretical decay curve

**Theory:** The gradient of the loss w.r.t. h₀ (the initial hidden state) is:
```
∂L/∂h₀ = ∂L/∂hT · ∏_{t=1}^{T} W_hh^T · diag(1 - h_t²)
```
If `spectral_radius(W_hh) < 1` or `tanh'(z_t) ≪ 1`, this product shrinks exponentially.

> 💡 **This is the core theoretical insight of the assignment.**
> A vanilla RNN with 14-day sequences works here because 14 is a manageable horizon.
> The same architecture would fail catastrophically on 60-day sequences.

In [ ]:
# ── TODO ─────────────────────────────────────────────────────────────
# ── Step 7: Gradient magnitude analysis ──────────────────────────────────────

# TODO 7a: Run one forward pass on a batch, compute loss, call backward()
# Then capture the gradient of the loss w.r.t. the hidden states at each step
# Hint: use register_hook to capture gradients on intermediate tensors
# OR: manually compute the gradient norm of W_hh power products

# Simplified approach (acceptable for this assignment):
# 1. Take one batch from train_dl
# 2. Do a forward pass with hook on each time-step output
# 3. Compute loss, do backward
# 4. Measure the gradient norms captured by the hooks

# YOUR CODE HERE (gradient capture and hook registration)

# TODO 7b: Plot gradient magnitude vs time step (from step T down to step 1)
# YOUR CODE HERE

# TODO 7c: Compute the spectral radius of W_hh
# spectral_radius = largest singular value of W_hh
# Print it and explain what it means for gradient flow
# YOUR CODE HERE

# TODO 7d: Write 2-sentence interpretation
# Interpretation:
raise NotImplementedError("Fill in the TODO above before continuing.")

---
## 📝 Step 8 — Evaluation & Visualisation

Evaluate on the held-out **test set** (never touched during training or validation).

**Required outputs:**
1. RMSE, MAE, MAPE, R² — in **original °C scale** (inverse-transform first)
2. Actual vs predicted temperature plot (full test period)
3. Scatter plot: actual vs predicted
4. Residual plot
5. Directional accuracy (did the model predict whether tomorrow is warmer or cooler?)

> 💡 **Inverse-transforming multivariate scaled data:**
> Your scaler scaled all features together. To get back to °C you need to
> inverse-transform only the target column. Build a dummy array of zeros,
> put your scaled predictions in the target column, inverse-transform the whole
> array, then extract just that column.

In [ ]:
# ── TODO ─────────────────────────────────────────────────────────────
# ── Step 8: Evaluation ────────────────────────────────────────────────────────

# TODO 8a: Generate test predictions and inverse-transform to °C
# Remember: scaler was fit on all FEATURE_COLS, so you need to inverse-transform
# using a dummy array trick (see guidance above)
# YOUR CODE HERE

# TODO 8b: Compute RMSE, MAE, MAPE, R²
# YOUR CODE HERE

# TODO 8c: Compute directional accuracy
# Sign of (actual[t] - actual[t-1]) vs sign of (predicted[t] - predicted[t-1])
# YOUR CODE HERE

# TODO 8d: Plot 1 — actual vs predicted temperature (line plot, full test period)
# YOUR CODE HERE

# TODO 8e: Plot 2 — scatter plot of actual vs predicted (+ perfect-fit diagonal)
# YOUR CODE HERE

# TODO 8f: Plot 3 — residuals over time
# YOUR CODE HERE

# TODO 8g: Write 3 bullet interpretations of your results
# Interpretation 1:
# Interpretation 2:
# Interpretation 3:
raise NotImplementedError("Fill in the TODO above before continuing.")

---
## 📝 Step 9 — Hyperparameter Tuning

Tune the two most impactful vanilla RNN hyperparameters.

**Grid to search:**
- `hidden_size` ∈ {16, 32, 64, 128}
- `SEQ_LEN` (lookback window) ∈ {7, 14, 21}

**Process:**
1. For each combination, train for 50 epochs (reduced for speed)
2. Record best validation MSE
3. Print a results table sorted by val MSE
4. Retrain the best configuration for 100 epochs

> 💡 **What hidden_size controls in a vanilla RNN:**
> `hidden_size` is the dimension of `h_t`. A larger hidden state can represent
> more complex patterns but has more parameters and takes longer to train.
> `n_params ≈ hidden_size × input_size + hidden_size² + hidden_size × output_size`
> The `hidden_size²` term (from `W_hh`) grows quadratically — be mindful.

> ⚠️ **For vanilla RNN, large hidden_size can worsen gradient problems**
> because `W_hh` becomes harder to keep stable. This is another reason LSTM/GRU
> are preferred in practice for large hidden dimensions.

In [ ]:
# ── TODO ─────────────────────────────────────────────────────────────
# ── Step 9: Hyperparameter grid search ───────────────────────────────────────

hidden_sizes = [16, 32, 64, 128]
seq_lengths  = [7, 14, 21]
results      = []

# TODO 9a: Grid search loop
# For each (hs, sl) pair:
#   1. Rebuild sequences with the new sl
#   2. Create new train_dl and val_dl
#   3. Instantiate fresh VanillaRNN(input_size=n_features, hidden_size=hs, ...)
#   4. Train for 50 epochs with patience=10
#   5. Record best val_loss
# YOUR CODE HERE

# TODO 9b: Print results table sorted by val MSE
# YOUR CODE HERE

# TODO 9c: Extract best config and retrain for 100 epochs
# YOUR CODE HERE
raise NotImplementedError("Fill in the TODO above before continuing.")

---
## 📝 Step 10 — Deployment Packaging

Package your trained model for production deployment.

**Deliverables:**
1. A `WeatherForecasterDeployment` class with `.predict()`, `.save()`, `.load()`
2. Three artefacts: `weather_model_weights.pt`, `weather_model_scaler.pkl`, `weather_model_config.json`
3. A simulated REST API function that accepts JSON and returns JSON
4. Load-from-disk verification

> 💡 **Production context:**
> In the real world this class would be deployed inside a **FastAPI** or **Flask** endpoint,
> containerised with **Docker**, and served on **AWS SageMaker**, **GCP Vertex AI**, or
> **Azure ML**. The `.pt` weights file would be stored in an MLflow or W&B model registry
> with version metadata (`created_at`, `val_rmse`, `seq_len`, etc.).

In [ ]:
# ── TODO ─────────────────────────────────────────────────────────────
# ── Step 10: Deployment Packaging ────────────────────────────────────────────

# TODO 10a: Implement WeatherForecasterDeployment class
# It must have:
#   __init__(self, model, scaler, config: dict)
#   predict(self, recent_data: np.ndarray) -> float
#     - recent_data: shape (seq_len, n_features) in ORIGINAL scale (°C etc.)
#     - returns: predicted next-day max temperature (°C)
#   save(self, prefix="weather_model")
#   classmethod load(cls, prefix, device="cpu")
# YOUR CODE HERE

# TODO 10b: Instantiate, populate config, and save
# config must include: seq_len, n_features, feature_cols, target_idx,
#                      hidden_size, num_layers, created_at, test_rmse
# YOUR CODE HERE

# TODO 10c: Simulate a REST API call
# def api_forecast(json_payload) -> dict:
#   payload:  {"city": "NYC", "features": [[...], ...]}   ← shape (seq_len, n_features)
#   response: {"city": "NYC", "predicted_max_temp_c": 22.4, "timestamp": "..."}
# YOUR CODE HERE

# TODO 10d: Load from disk and verify predictions match (delta < 1e-5°C)
# YOUR CODE HERE
raise NotImplementedError("Fill in the TODO above before continuing.")

---
## 📝 Step 11 — Reflection Questions

Answer each question in the cell below it. Be specific — reference your results.

---

### Q1. Vanilla RNN vs LSTM: in this weather problem, would LSTM give significantly better results? Why or why not?

*(Write your answer here)*

---

### Q2. Why is gradient clipping mandatory for vanilla RNNs but not strictly required for LSTMs?

*(Write your answer here — reference the equations)*

---

### Q3. Your model uses `SEQ_LEN = 14`. From your ACF plot (Step 2), is 14 days sufficient or would a longer window help?

*(Write your answer here — cite the ACF values)*

---

### Q4. The `W_hh` matrix has shape `(hidden_size × hidden_size)`. Explain why its spectral radius matters for training stability.

*(Write your answer here)*

---

### Q5. Name two failure modes specific to deploying a weather forecasting RNN in production that would NOT occur in a tabular/non-sequential model.

*(Write your answer here)*